<!-- placeholder (notebook intro is in next cell) -->

# Phase 1 - Notebook 1: Driving-Style Validation (legacy 4-feature K-means)

**Goal.** Decide whether the legacy K-means styles (clustering on `[max_abs_accel, var_speed, var_acceleration, gamma]`) produce *meaningful* aggressive / normal (/ cautious) clusters on cleaned AV2 **train**.

**Train-only decision.** Phase 1 uses the train split for all decisions because the downstream use of the difficulty score is training-time reweighting/resampling on train. Val numbers are reported as an appendix / diagnostic only and do not feed the verdict. A train->val generalization check belongs in Phase 2.

This notebook is a **soft gate**: failure here triggers PIVOT (drop style as central contribution), not STOP. Pre-registered thresholds live in `docs/phase1_decision_criteria.md`.

> **Known limitation:** the aggressive cluster is chosen by `highest mean max_abs_accel`, which is speed-correlated. High-speed agents on highways can therefore be labelled aggressive even when their driving is unremarkable for the scene. Scene-normalized style naming is deferred to Phase 1.5.

Expected inputs (produced by `scripts/run_difficulty_analysis.py --phase1`):

- `artifacts/phase1/tables/av2_train_feature_table.csv`
- `artifacts/phase1/tables/av2_val_feature_table.csv` (for val appendix / GIFs only)

Outputs written here:

- `artifacts/phase1/style_model_K{2,3}.pkl`
- `artifacts/phase1/cleaning_report_{train,val}.json`
- `artifacts/phase1/style_label_map.yaml` (label ordering derived from **train**)
- `artifacts/phase1/cleaned_train_with_style.parquet` - canonical input to NB2 and NB3; includes `style_cluster_id_K{2,3}`, `style_label_K{2,3}`, `style_cluster_is_aggressive_K{2,3}` (0/1)
- `artifacts/phase1/cleaned_val_with_style.parquet` - appendix / GIF notebook input; no decision depends on it
- `artifacts/phase1/figures/nb1_*.png`

## Verdict cell (filled in by the final cell)

**Style meaningfulness (on TRAIN):** _TBD - see final cell._

Sub-checks (all evaluated on train):
- Behavioral extrema (|Cohen's d| >= 0.5 on >= 3 extrema, sign-consistent K=2 vs K=3): _TBD_
- Rule-based F1 >= 0.35 at K=2 and K=3: _TBD_

In [ ]:
from __future__ import annotations
import sys, json, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if (Path.cwd().name == 'notebooks') else Path.cwd()
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from tailrisk_mp.cleaning import CLUSTERING_FEATURES, CleaningConfig, filter_for_clustering

ARTIFACTS = REPO_ROOT / 'artifacts' / 'phase1'
TABLES = ARTIFACTS / 'tables'
FIG_DIR = ARTIFACTS / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = TABLES / 'av2_train_feature_table.csv'
VAL_CSV   = TABLES / 'av2_val_feature_table.csv'

warnings.filterwarnings('ignore', category=FutureWarning)
print('Repo root:', REPO_ROOT)
print('Train table exists:', TRAIN_CSV.exists())
print('Val   table exists:', VAL_CSV.exists())


def _load(path):
    if not path.exists():
        print('MISSING:', path)
        print('Run: scripts/run_on_gpu_node.sh python scripts/run_difficulty_analysis.py --phase1 --datasets av2')
        return pd.DataFrame()
    df = pd.read_csv(path)
    print('Loaded', path.name, len(df), 'rows', df.shape[1], 'cols')
    return df

train_df = _load(TRAIN_CSV)
val_df   = _load(VAL_CSV)

for name, df in [('train', train_df), ('val', val_df)]:
    if not df.empty:
        missing = [c for c in (*CLUSTERING_FEATURES, 'is_stationary', 'n_valid_future_steps', 'center_objects_type') if c not in df.columns]
        if missing:
            print('WARNING:', name, 'missing columns:', missing)

## 1. Pre-cluster cleaning (`filter_for_clustering`)

Runs the cleaning module on train, fits percentile clips + Mahalanobis envelope, then applies the *same* cleaner to val. Drop reasons: stationary, short track (<30 valid future steps), non-vehicle, NaN feature, out-of-percentile, Mahalanobis outlier.

In [ ]:
cleaned_train, report_train = pd.DataFrame(), {}
cleaned_val, report_val = pd.DataFrame(), {}
if not train_df.empty:
    cfg_train = CleaningConfig(fit_stats=True)
    cleaned_train, report_train = filter_for_clustering(
        train_df, cfg_train, report_path=ARTIFACTS / 'cleaning_report_train.json'
    )
if not val_df.empty and not cleaned_train.empty:
    cfg_val = CleaningConfig(fit_stats=False, reuse_stats=report_train['clean_stats'])
    cleaned_val, report_val = filter_for_clustering(
        val_df, cfg_val, report_path=ARTIFACTS / 'cleaning_report_val.json'
    )

def _drop_table(rep):
    if not rep: return pd.DataFrame()
    rows = []
    for k, v in rep.get('drop_counts', {}).items():
        rows.append({'reason': k, 'count': v, 'fraction': rep.get('drop_fractions', {}).get(k, 0.0)})
    rows.append({'reason': 'KEPT', 'count': rep.get('kept_rows', 0), 'fraction': rep.get('kept_fraction', 0.0)})
    return pd.DataFrame(rows)

print('=== Train drop table ===')
print(_drop_table(report_train).to_string(index=False))
print('\n=== Val drop table ===')
print(_drop_table(report_val).to_string(index=False))

# Figures: stationary share + before/after boxplots on the 4 clustering features.
if not train_df.empty:
    if 'is_stationary' in train_df.columns:
        fig, ax = plt.subplots(1, 1, figsize=(5, 4))
        counts = train_df['is_stationary'].astype(bool).value_counts()
        ax.pie(counts, labels=['stationary=' + str(k) for k in counts.index], autopct='%1.1f%%', startangle=90)
        ax.set_title('Train: stationary share')
        fig.tight_layout(); fig.savefig(FIG_DIR / 'nb1_cleaning_stationary.png', dpi=120, bbox_inches='tight')
        plt.show()

    fig2, axes = plt.subplots(1, len(CLUSTERING_FEATURES), figsize=(4 * len(CLUSTERING_FEATURES), 3.5))
    for ax, f in zip(axes, CLUSTERING_FEATURES):
        data = [train_df[f].dropna().values if f in train_df.columns else [],
                cleaned_train[f].dropna().values if f in cleaned_train.columns else []]
        ax.boxplot(data, labels=['raw', 'clean'], showfliers=False)
        ax.set_title(f)
    fig2.suptitle('Clustering features before vs after cleaning (train)')
    fig2.tight_layout(); fig2.savefig(FIG_DIR / 'nb1_cleaning_boxplots.png', dpi=120, bbox_inches='tight')
    plt.show()

## 2. Fit clustering (K=2, K=3) on cleaned train, apply to cleaned val

`StandardScaler` + `MiniBatchKMeans(n_init=20)` on the 4 legacy features. Fit on train only; apply to val. Save (scaler, model) so Notebook 1b / 2 / 3 reuse exactly the same style assignment.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans

style_models = {}
if not cleaned_train.empty:
    Xtr = cleaned_train[list(CLUSTERING_FEATURES)].to_numpy(dtype=np.float64)
    scaler = StandardScaler().fit(Xtr)
    Xtr_s = scaler.transform(Xtr)
    for K in (2, 3):
        km = MiniBatchKMeans(n_clusters=K, random_state=42, n_init=20,
                             batch_size=min(2048, max(len(Xtr_s), 256)))
        labels_tr = km.fit_predict(Xtr_s)
        dist_tr = np.linalg.norm(Xtr_s - km.cluster_centers_[labels_tr], axis=1)
        cleaned_train['style_cluster_id_K' + str(K)] = labels_tr
        cleaned_train['style_cluster_distance_K' + str(K)] = dist_tr
        style_models[K] = {'scaler': scaler, 'model': km}
        with open(ARTIFACTS / ('style_model_K' + str(K) + '.pkl'), 'wb') as f:
            pickle.dump(style_models[K], f)

if not cleaned_val.empty and style_models:
    Xv = cleaned_val[list(CLUSTERING_FEATURES)].to_numpy(dtype=np.float64)
    Xv_s = style_models[2]['scaler'].transform(Xv)
    for K in (2, 3):
        labels_v = style_models[K]['model'].predict(Xv_s)
        dist_v = np.linalg.norm(Xv_s - style_models[K]['model'].cluster_centers_[labels_v], axis=1)
        cleaned_val['style_cluster_id_K' + str(K)] = labels_v
        cleaned_val['style_cluster_distance_K' + str(K)] = dist_v

for K in (2, 3):
    col = 'style_cluster_id_K' + str(K)
    if col in cleaned_val.columns:
        counts = cleaned_val[col].value_counts().sort_index()
        print('Val cluster sizes K=' + str(K) + ':')
        print(counts.to_string())

In [ ]:
import seaborn as sns

In [ ]:
for K in (2, 3):
    col = 'style_cluster_id_K' + str(K)
    if col in cleaned_train.columns:
        counts = cleaned_train[col].value_counts().sort_index()
        print('Train cluster sizes K=' + str(K) + ':')
        print(counts.to_string())

In [ ]:
sns.histplot(cleaned_train, x='avg_speed', hue='style_cluster_id_K2', stat='percent', common_norm=False)
plt.show()
sns.histplot(cleaned_train, x='avg_speed', hue='style_cluster_id_K3', stat='percent', common_norm=False)
plt.show()

In [ ]:
sns.histplot(cleaned_val, x='avg_speed', hue='style_cluster_id_K2', stat='percent', common_norm=False)
plt.show()
sns.histplot(cleaned_val, x='avg_speed', hue='style_cluster_id_K3', stat='percent', common_norm=False)
plt.show()

## 3. Stability across seeds (ARI) + 4. Internal metrics (silhouette, DB, CH, elbow)

In [ ]:
from itertools import combinations
from sklearn.metrics import (adjusted_rand_score, silhouette_score,
                             davies_bouldin_score, calinski_harabasz_score)

stability_rows = []
internal_rows = []
if not cleaned_train.empty:
    Xtr_s = style_models[2]['scaler'].transform(cleaned_train[list(CLUSTERING_FEATURES)].to_numpy())
    for K in (2, 3):
        labels_across = []
        for seed in range(10):
            km = MiniBatchKMeans(n_clusters=K, random_state=seed, n_init=5,
                                 batch_size=min(2048, max(len(Xtr_s), 256)))
            labels_across.append(km.fit_predict(Xtr_s))
        aris = [adjusted_rand_score(a, b) for a, b in combinations(labels_across, 2)]
        stability_rows.append({'K': K, 'ari_mean': float(np.mean(aris)),
                               'ari_median': float(np.median(aris)),
                               'ari_min': float(np.min(aris)), 'ari_max': float(np.max(aris))})
    sample_idx = np.random.default_rng(0).choice(len(Xtr_s), size=min(10000, len(Xtr_s)), replace=False)
    X_sub = Xtr_s[sample_idx]
    for K in (2, 3, 4, 5):
        km = MiniBatchKMeans(n_clusters=K, random_state=42, n_init=10,
                             batch_size=min(2048, max(len(Xtr_s), 256)))
        lbl = km.fit_predict(Xtr_s)
        internal_rows.append({
            'K': K, 'inertia': float(km.inertia_),
            'silhouette_sub': float(silhouette_score(X_sub, lbl[sample_idx])) if len(set(lbl[sample_idx])) > 1 else float('nan'),
            'davies_bouldin': float(davies_bouldin_score(Xtr_s, lbl)),
            'calinski_harabasz': float(calinski_harabasz_score(Xtr_s, lbl)),
        })

print('=== Stability (ARI across seeds) ===')
print(pd.DataFrame(stability_rows).to_string(index=False))
print('\n=== Internal metrics ===')
internal_df = pd.DataFrame(internal_rows)
print(internal_df.to_string(index=False))

if not internal_df.empty:
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
    for ax, col in zip(axes, ['inertia', 'silhouette_sub', 'davies_bouldin', 'calinski_harabasz']):
        ax.plot(internal_df['K'], internal_df[col], marker='o')
        ax.set_xlabel('K'); ax.set_title(col)
    fig.tight_layout(); fig.savefig(FIG_DIR / 'nb1_internal_metrics.png', dpi=120, bbox_inches='tight')
    plt.show()

## 5. Per-cluster kinematic distributions (speed + acceleration + jerk)

Overlaid histograms on shared x-axes + compact violin plots. These are the key "does aggressive actually look aggressive?" figures.

In [ ]:
KIN_FEATURES = ['avg_speed', 'max_speed', 'var_speed',
                'avg_accel', 'max_abs_accel', 'var_acceleration',
                'avg_jerk', 'max_abs_jerk', 'gamma']


def _plot_cluster_histograms(df, K, features, out_path, split_name):
    col = 'style_cluster_id_K' + str(K)
    if col not in df.columns:
        return
    n = len(features); cols_ = 3
    rows = int(np.ceil(n / cols_))
    fig, axes = plt.subplots(rows, cols_, figsize=(4.5 * cols_, 3.0 * rows))
    axes = np.atleast_2d(axes).reshape(rows, cols_)
    clusters = sorted(df[col].dropna().unique())
    cmap = plt.get_cmap('tab10')
    for idx, feat in enumerate(features):
        ax = axes[idx // cols_, idx % cols_]
        if feat not in df.columns:
            ax.set_visible(False); continue
        values_all = pd.to_numeric(df[feat], errors='coerce')
        lo, hi = np.nanpercentile(values_all, [1, 99])
        bins = np.linspace(lo, hi, 50)
        for j, c in enumerate(clusters):
            vals = pd.to_numeric(df[df[col] == c][feat], errors='coerce').dropna()
            if len(vals) == 0:
                continue
            ax.hist(vals.clip(lo, hi), bins=bins, alpha=0.45,
                    label='c' + str(c), color=cmap(j), density=True)
        ax.set_title(feat)
        ax.legend(fontsize=7)
    for k in range(n, rows * cols_):
        axes[k // cols_, k % cols_].set_visible(False)
    fig.suptitle(split_name + ' per-cluster kinematics (K=' + str(K) + ')')
    fig.tight_layout(); fig.savefig(out_path, dpi=120, bbox_inches='tight'); plt.show()


def _plot_cluster_violins(df, K, features, out_path, split_name):
    col = 'style_cluster_id_K' + str(K)
    if col not in df.columns:
        return
    clusters = sorted(df[col].dropna().unique())
    fig, axes = plt.subplots(1, len(features), figsize=(2.4 * len(features), 3.5))
    for ax, feat in zip(axes, features):
        data = [pd.to_numeric(df[df[col] == c][feat], errors='coerce').dropna().values
                for c in clusters]
        ax.violinplot(data, showmeans=True, showextrema=False)
        ax.set_xticks(range(1, len(clusters) + 1))
        ax.set_xticklabels(['c' + str(c) for c in clusters])
        ax.set_title(feat, fontsize=9)
    fig.suptitle(split_name + ' violins (K=' + str(K) + ')')
    fig.tight_layout(); fig.savefig(out_path, dpi=120, bbox_inches='tight'); plt.show()


for K in (2, 3):
    if not cleaned_val.empty:
        _plot_cluster_histograms(cleaned_train, K, KIN_FEATURES,
                                 FIG_DIR / ('nb1_hist_val_K' + str(K) + '.png'), 'Train')
        _plot_cluster_violins(cleaned_train, K,
                              ['avg_speed', 'max_speed', 'avg_accel', 'max_abs_accel', 'avg_jerk'],
                              FIG_DIR / ('nb1_violin_val_K' + str(K) + '.png'), 'Train')

## 6. Primary validation A - behavioral extrema + Cohen's d (TRAIN)

The aggressive cluster (highest mean `max_abs_accel`) vs the rest on extrema features **not** part of the clustering input. Pass iff |d| >= 0.5 on >= 3 extrema, sign-consistent between K=2 and K=3.

**Train is the decision source.** Val tables are shown afterwards for diagnostic purposes only and are NOT used in the verdict.

In [ ]:
EXTREMA = ['hard_brake_count', 'hard_accel_count', 'lateral_g_spike_count',
           'high_jerk_count', 'heading_rate_p95', 'max_abs_lat_accel',
           'max_closing_speed', 'inverse_min_ttc', 'close_encounter_count']


def cohens_d(a, b):
    a = np.asarray(a, dtype=np.float64); b = np.asarray(b, dtype=np.float64)
    a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
    if len(a) < 2 or len(b) < 2: return float('nan')
    na, nb = len(a), len(b); va, vb = np.var(a, ddof=1), np.var(b, ddof=1)
    pooled = np.sqrt(((na - 1) * va + (nb - 1) * vb) / (na + nb - 2))
    if pooled < 1e-9: return float('nan')
    return float((np.mean(a) - np.mean(b)) / pooled)


def extrema_table(df, K):
    col = 'style_cluster_id_K' + str(K)
    if col not in df.columns or df.empty: return pd.DataFrame()
    feats = [f for f in EXTREMA if f in df.columns]
    if not feats: return pd.DataFrame()
    means = df.groupby(col)[feats].mean()
    if 'max_abs_accel' in df.columns:
        most_agg = int(df.groupby(col)['max_abs_accel'].mean().idxmax())
    else:
        most_agg = int(means.index[0])
    rows = []
    agg_mask = df[col] == most_agg
    for feat in means.columns:
        d = cohens_d(df.loc[agg_mask, feat].to_numpy(), df.loc[~agg_mask, feat].to_numpy())
        rows.append({'feature': feat, 'aggressive_cluster': most_agg,
                     'mean_agg': float(df.loc[agg_mask, feat].mean()),
                     'mean_other': float(df.loc[~agg_mask, feat].mean()),
                     'cohens_d': d})
    return pd.DataFrame(rows).sort_values('cohens_d', key=lambda s: s.abs(), ascending=False)


# Primary (decision-relevant): TRAIN split only.
extr_K2 = extrema_table(cleaned_train, 2)
extr_K3 = extrema_table(cleaned_train, 3)
print('=== K=2 extrema table (TRAIN, decision-relevant) ==='); print(extr_K2.to_string(index=False))
print('\n=== K=3 extrema table (TRAIN, decision-relevant) ==='); print(extr_K3.to_string(index=False))

n_pass_K2 = int((extr_K2['cohens_d'].abs() >= 0.5).sum()) if not extr_K2.empty else 0
n_pass_K3 = int((extr_K3['cohens_d'].abs() >= 0.5).sum()) if not extr_K3.empty else 0
if not extr_K2.empty and not extr_K3.empty:
    merged = extr_K2.merge(extr_K3, on='feature', suffixes=('_K2', '_K3'))
    sign_consistent = int(((merged['cohens_d_K2'].fillna(0) > 0) == (merged['cohens_d_K3'].fillna(0) > 0)).sum())
else:
    sign_consistent = 0
extrema_pass = (n_pass_K2 >= 3) and (n_pass_K3 >= 3) and (sign_consistent >= 3)
print('TRAIN |d|>=0.5 count K2=' + str(n_pass_K2) + ', K3=' + str(n_pass_K3) +
      '; sign-consistent features=' + str(sign_consistent) +
      ' => EXTREMA_PASS=' + str(extrema_pass))

# Appendix (diagnostic only): val extrema. NOT used in the verdict.
print('\n--- appendix (val, diagnostic only, NOT used in verdict) ---')
extr_K2_val = extrema_table(cleaned_val, 2)
extr_K3_val = extrema_table(cleaned_val, 3)
if not extr_K2_val.empty:
    print('=== K=2 extrema table (val) ==='); print(extr_K2_val.to_string(index=False))
if not extr_K3_val.empty:
    print('\n=== K=3 extrema table (val) ==='); print(extr_K3_val.to_string(index=False))

## 7. Primary validation B - rule-based aggressive label + precision/recall/F1 (TRAIN)

`rule_aggressive = hard_brake | hard_accel | high_jerk | lateral_g_spike`. Precision / recall / F1 / balanced accuracy / PR-AUC of (cluster == candidate_aggressive) vs the rule. PASS iff F1 >= 0.35 at K=2 and K=3, **evaluated on train**.

Also reports sensitivity under 3 alternative rule definitions (loose / strict / OR-of-rates) so the F1 is not a single fragile number. Val numbers are appendix-only.

In [ ]:
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             balanced_accuracy_score, average_precision_score)


def _rule_aggressive(df, *, mode='primary'):
    cols = ['hard_brake_count', 'hard_accel_count', 'high_jerk_count', 'lateral_g_spike_count']
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print('rule: missing cols', missing)
        return pd.Series(False, index=df.index)
    counts = df[cols].fillna(0).astype(float)
    if mode == 'primary':
        return (counts >= 1).any(axis=1)
    if mode == 'loose':
        return counts.sum(axis=1) >= 1
    if mode == 'strict':
        return (counts >= 1).sum(axis=1) >= 2
    if mode == 'rate':
        dur = pd.to_numeric(df.get('track_duration_s', 1.0), errors='coerce').fillna(1.0).clip(lower=0.5)
        rates = counts.div(dur, axis=0)
        return (rates >= 0.25).any(axis=1)
    raise ValueError('unknown mode ' + mode)


def _rule_metrics(df, *, K, rule_mode):
    col = 'style_cluster_id_K' + str(K)
    if df.empty or col not in df.columns:
        return None
    rule = _rule_aggressive(df, mode=rule_mode)
    prevalence = float(rule.mean())
    most_agg = int(df.groupby(col)['max_abs_accel'].mean().idxmax()) \
        if 'max_abs_accel' in df.columns else 0
    pred = df[col] == most_agg
    if rule.sum() == 0 or pred.sum() == 0:
        return {'rule': rule_mode, 'K': K, 'prevalence': prevalence,
                'candidate_cluster': most_agg,
                'precision': np.nan, 'recall': np.nan, 'f1': np.nan,
                'balanced_acc': np.nan, 'pr_auc': np.nan}
    return {
        'rule': rule_mode, 'K': K, 'prevalence': prevalence,
        'candidate_cluster': most_agg,
        'precision': float(precision_score(rule, pred)),
        'recall':    float(recall_score(rule, pred)),
        'f1':        float(f1_score(rule, pred)),
        'balanced_acc': float(balanced_accuracy_score(rule, pred)),
        'pr_auc':    float(average_precision_score(rule.astype(int), pred.astype(int))),
    }


# Primary (decision-relevant): TRAIN split only.
rule_rows = []
for rule_mode in ('primary', 'loose', 'strict', 'rate'):
    for K in (2, 3):
        row = _rule_metrics(cleaned_train, K=K, rule_mode=rule_mode)
        if row is not None:
            rule_rows.append(row)

rule_df = pd.DataFrame(rule_rows)
print('=== TRAIN rule-based F1 (decision-relevant) ===')
print(rule_df.to_string(index=False))
primary = rule_df[rule_df.get('rule', '') == 'primary'] if not rule_df.empty else rule_df
rule_pass = bool(not primary.empty and (primary['f1'] >= 0.35).all())
print('TRAIN RULE_PASS (F1 >= 0.35 at K=2 AND K=3, primary rule) =', rule_pass)

# Appendix (diagnostic only): val rule-based F1. NOT used in the verdict.
val_rule_rows = []
for rule_mode in ('primary', 'loose', 'strict', 'rate'):
    for K in (2, 3):
        row = _rule_metrics(cleaned_val, K=K, rule_mode=rule_mode)
        if row is not None:
            val_rule_rows.append(row)

val_rule_df = pd.DataFrame(val_rule_rows)
if not val_rule_df.empty:
    print('\n--- appendix (val, diagnostic only, NOT used in verdict) ---')
    print(val_rule_df.to_string(index=False))

## 8. Label map + TDBM / trajectory-type cross-tabs

Persist a deterministic label map (cluster id -> {aggressive, normal, cautious}) built from **train** so downstream notebooks always name clusters the same way. Also derive `style_cluster_is_aggressive_K{2,3}` 0/1 flags that NB3's handcrafted composite consumes directly. Cross-tab K-means labels against `tdbm_primary_behavior` and `trajectory_type` on train to sanity-check the assignment.

In [ ]:
import yaml

# Build label map from TRAIN (decision-relevant), then propagate the same mapping to val.
label_map = {}
if not cleaned_train.empty and 'max_abs_accel' in cleaned_train.columns:
    for K in (2, 3):
        col = 'style_cluster_id_K' + str(K)
        if col not in cleaned_train.columns: continue
        means = cleaned_train.groupby(col)['max_abs_accel'].mean().sort_values()
        if K == 2:
            names = ['normal', 'aggressive']
        else:
            names = ['cautious', 'normal', 'aggressive']
        mapping = {int(cid): nm for cid, nm in zip(means.index, names)}
        label_map['K' + str(K)] = mapping
        cleaned_train['style_label_K' + str(K)] = cleaned_train[col].map(mapping)
        # Boolean "is_aggressive" flag used by the NB3 handcrafted composite.
        cleaned_train['style_cluster_is_aggressive_K' + str(K)] = (
            cleaned_train['style_label_K' + str(K)] == 'aggressive'
        ).astype(int)
        if not cleaned_val.empty and col in cleaned_val.columns:
            cleaned_val['style_label_K' + str(K)] = cleaned_val[col].map(mapping)
            cleaned_val['style_cluster_is_aggressive_K' + str(K)] = (
                cleaned_val['style_label_K' + str(K)] == 'aggressive'
            ).astype(int)

label_yaml = ARTIFACTS / 'style_label_map.yaml'
with open(label_yaml, 'w') as f:
    yaml.safe_dump({'sort_feature': 'max_abs_accel', 'source_split': 'train', 'mapping': label_map}, f)
print('Wrote', label_yaml)
print(json.dumps(label_map, indent=2))

print('\n=== TRAIN crosstabs (decision-relevant) ===')
for K in (2, 3):
    lbl = 'style_label_K' + str(K)
    if lbl not in cleaned_train.columns: continue
    for reference in ('tdbm_primary_behavior', 'trajectory_type', 'center_objects_type'):
        if reference not in cleaned_train.columns: continue
        ct = pd.crosstab(cleaned_train[lbl], cleaned_train[reference], normalize='index')
        print('\n=== TRAIN K=' + str(K) + ' x ' + reference + ' (row-normalized) ===')
        print(ct.round(3).to_string())

## 9. Persist cleaned tables with style labels

Save `cleaned_train_with_style.parquet` and `cleaned_val_with_style.parquet` (both splits get `style_cluster_id_K{2,3}`, `style_cluster_distance_K{2,3}`, `style_label_K{2,3}`). Notebooks 1b / 2 / 3 load these directly.

In [ ]:
OUT_TRAIN = ARTIFACTS / 'cleaned_train_with_style.parquet'
OUT_VAL   = ARTIFACTS / 'cleaned_val_with_style.parquet'

# Sanity check: NB3 consumes these columns.
required_cols = [
    'style_cluster_id_K2', 'style_cluster_id_K3',
    'style_label_K2', 'style_label_K3',
    'style_cluster_is_aggressive_K2', 'style_cluster_is_aggressive_K3',
]
missing = [c for c in required_cols if c not in cleaned_train.columns]
if missing:
    print('WARNING: cleaned_train is missing expected columns:', missing)
else:
    print('cleaned_train has all style columns NB3 needs.')


def _safe_parquet(df, path):
    if df.empty:
        print('skip (empty):', path); return
    try:
        df.to_parquet(path, index=False)
    except Exception as err:
        csv_path = path.with_suffix('.csv')
        print('parquet failed (' + str(err) + ') - falling back to CSV at', csv_path)
        df.to_csv(csv_path, index=False)
        return
    print('Wrote', path, df.shape)

_safe_parquet(cleaned_train, OUT_TRAIN)
_safe_parquet(cleaned_val, OUT_VAL)  # appendix / GIF notebook only; no decision depends on it.

## 10. Verdict

Combines the two sub-checks into a single decision. Update the top-of-notebook verdict cell with these values before committing.

- EXTREMA_PASS: >= 3 extrema with |Cohen's d| >= 0.5, sign-consistent K=2 vs K=3.
- RULE_PASS: primary-rule F1 >= 0.35 at K=2 AND K=3.

Decision (also documented in `docs/phase1_decision_criteria.md`):

- Both pass => GO (style is meaningful, keep as Phase 1 feature).
- One passes => SOFT-PASS (style kept as auxiliary feature, not headline contribution).
- Both fail => PIVOT (drop style as a core contribution; use kinematics + social + extrema only).

In [ ]:
verdict = {
    'decision_source': 'train',
    'extrema_pass': bool(extrema_pass),
    'rule_pass': bool(rule_pass),
    'n_extrema_pass_K2': int(n_pass_K2),
    'n_extrema_pass_K3': int(n_pass_K3),
    'sign_consistent_features': int(sign_consistent),
    'rule_f1_train': rule_df.to_dict(orient='records') if not rule_df.empty else [],
    'rule_f1_val_appendix': val_rule_df.to_dict(orient='records') if not val_rule_df.empty else [],
    'stability_ari': stability_rows,
    'internal_metrics': internal_rows,
    'label_map': label_map,
}
if extrema_pass and rule_pass:
    verdict['decision'] = 'GO'
elif extrema_pass or rule_pass:
    verdict['decision'] = 'SOFT-PASS'
else:
    verdict['decision'] = 'PIVOT'

verdict_path = ARTIFACTS / 'nb1_verdict.json'
with open(verdict_path, 'w') as f:
    json.dump(verdict, f, indent=2, default=float)
print('Wrote', verdict_path)
print(json.dumps({k: v for k, v in verdict.items() if k in ('decision', 'extrema_pass', 'rule_pass',
                                                             'n_extrema_pass_K2', 'n_extrema_pass_K3',
                                                             'sign_consistent_features')}, indent=2))